<a href="https://colab.research.google.com/github/saad0O5/FlyRank-Internship-Work/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saad0O5/FlyRank-s-Project/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Scoring / Ranking (not plain classification, though the underlying label is binary).**

The lane's real output is a ranked priority queue — pages ordered by decline risk so a reviewer with limited capacity works top-down. A binary classifier alone (predict "declining" yes/no) doesn't fit the actual decision: reviewers don't act on a threshold, they act on a capacity-limited list. So I train a classifier (is_declining) but consume its probability output as a ranking score, not its hard label. This is scoring-for-ranking, the same shape as Lane 2 in the guide.


In [1]:
!pip install -q reportlab

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/saad0O5/FlyRank-Internship-Work"
REPO_DIR = "FlyRank-Internship-Work"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
print(df.shape)

(30000, 45)


In [3]:
!{sys.executable} scripts/run_all.py


▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/FlyRank-Internship-Work/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/FlyRank-Internship-Work/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/FlyRank-Internship-Work/data/processed/model_predictions.csv
Wrote model results: /content/FlyRank-Internship-Work/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queue, charts, and the Markdown report
Wrote final refresh queue: /content/FlyRank-Internship-Work/outputs/refresh_queue.csv
Wrote model report: /co

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: is_declining = (trend_direction == "down"). This is a proxy label, not an observed future outcome — it's a bucket computed from the current 90-day window, so it describes present state rather than "will this page decline next month." I'm using it because it's what the starter dataset ships. Section 5 of the lane guide flags this exact weakness: a stronger capstone version would use a genuine prior-window → future-window label (e.g. prior 90 days of features predicting next-30-day decline) once I move to the warehouse release, where daily time-series data makes that possible. For now, the proxy is honestly labeled as a proxy, not treated as ground truth.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(df["trend_direction"].value_counts())
print(f"\nis_declining rate: {df['is_declining'].mean():.3f}")

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

is_declining rate: 0.542


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Metric: Precision@50 (or Precision@K generally, K = reviewer capacity). Overall accuracy is the wrong metric here — the reviewer never sees the whole ranked list, only the top K, so what matters is "of the top 50 the model flags, how many are real." This directly matches the cost structure: a false positive near the top wastes review time; a true decline buried below K is never seen. Precision@50 already has a number I can defend from Week 1: baseline rule = 0.240, random forest = 0.740, a ~3.1x lift.

In [5]:
import json
res = json.load(open("outputs/model_results.json"))

base = res["baseline"]["baseline_precision_at_50"]
rf = res["models"]["random_forest"]["precision_at_50"]
print(f"Baseline Precision@50: {base:.3f}   Random Forest Precision@50: {rf:.3f}   ({rf/base:.1f}x lift)")

Baseline Precision@50: 0.240   Random Forest Precision@50: 0.740   (3.1x lift)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content item (content_id), deduplicated, with impressions_90d > 0 and content_age_days >= 90 already filtered by the starter pipeline. Each row carries observable signals (impressions, clicks, position, CTR, age, content_type, client_id) plus the derived is_declining proxy target.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Shape: {df.shape}  ({df['content_id'].nunique()} unique content_ids)")
df[["content_id", "client_id", "content_type", "content_age_days",
    "impressions_90d", "trend_direction", "is_declining"]].head()

Shape: (30000, 45)  (30000 unique content_ids)


,content_id,client_id,content_type,content_age_days,impressions_90d,trend_direction,is_declining
0,content_304f48230142,client_f369cb89fc,keyword article,187,3803,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,15320,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,12581,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,11751,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,19140,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (e.g. "flag pages older than X days" or "flag pages with CTR below Y") can't capture what Week 1 already showed: the signals are tangled and partly independent. content_age_days correlates only weakly with decline (r = -0.164, and in the opposite direction a pure staleness rule would assume). content_type has a large, separate effect (comparison/keyword articles decline at ~double the rate of feedly articles) that holds across age tiers, so it isn't just age wearing a different name. And decline is unevenly concentrated by client (0% to 93.7%), which a global threshold rule can't account for at all. A learned model can weigh and combine these interacting, non-linear effects — a hand-written if/else rule can express at most one or two of them at a time, which is exactly why the baseline rule tops out at Precision@50 = 0.240 while the random forest reaches 0.740 on the same data.

In [8]:
corr_age = df["content_age_days"].corr(df["is_declining"])
print(f"Correlation between content_age_days and is_declining: {corr_age:.3f}")

print(df.groupby("content_type")["is_declining"].mean().round(3))
client_decline = df.groupby("client_id")["is_declining"].mean()
print(f"\nClient decline rate range: {client_decline.min():.3f} to {client_decline.max():.3f}")

Correlation between content_age_days and is_declining: -0.164
content_type
comparison article    0.572
feedly article        0.287
keyword article       0.561
Name: is_declining, dtype: float64

Client decline rate range: 0.000 to 0.937


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.